# 02 — Feature Extraction Pipeline (Multiprocessing)

**Objective:** Extract 33 handcrafted forensic features from 87,971 preprocessed `.npy` arrays  
and produce a single `features_dataset.csv` with **split_role provenance** ready for model training.

| Group | Module | Features | Domain |
|-------|--------|----------|--------|
| 1 | `frequency_features` | 6 | FFT power spectrum, DCT mid-band stats |
| 2 | `color_features` | 9 | Cross-channel correlation, GLCM texture |
| 3 | `microtexture_features` | 10 | SRM residuals, chroma LBP |
| 4 | `spatial_features` | 8 | SNR ratio, flat-zone skew/kurtosis |

**Architecture:**
- Each `.npy` file is read from disk **once** → fed to all 4 extractors (I/O optimized)
- `ProcessPoolExecutor` bypasses the GIL for CPU-bound NumPy/SciPy operations
- Per-file error isolation: failures → `status="error"` with NaN features, never crash the batch
- Group 4 outputs may contain `np.nan` (physical dead-signal) → preserved for downstream Dual-Imputation

**Anti-leakage design:** Every row is tagged with `split_role` ∈ {`train_core`, `calibration`, `val`, `id_test`, `ood_eval`}.  
OOD generators (SDv15, GLIDE) are **never** mixed into train/val/calibration splits.  
Sanity checks in Cell 6 compute feature statistics **only on `train_core`** — the same fence that will govern  
imputation fit, feature selection, and outlier thresholds downstream.

## Cell 1 — Environment & Imports

In [7]:
"""Cell 1 — Environment & Imports."""

from __future__ import annotations

import logging
import os
import sys
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ── Project root on sys.path so `src.*` imports resolve ──────────────
PROJECT_ROOT = Path.cwd().parent  # notebooks/ → project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Worker function lives in a module (required for Windows ProcessPoolExecutor)
from src.feature_extraction.worker import extract_all_features, ALL_FEATURE_KEYS

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("02_feature_extraction")
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Feature key groups (for sanity checks) ────────────────────────────
from src.feature_extraction.frequency import FEATURE_KEYS as FREQ_KEYS
from src.feature_extraction.color import FEATURE_KEYS as COLOR_KEYS
from src.feature_extraction.microtexture import FEATURE_KEYS as MICRO_KEYS
from src.feature_extraction.spatial import FEATURE_KEYS as SPATIAL_KEYS

assert len(ALL_FEATURE_KEYS) == 33, f"Expected 33 features, got {len(ALL_FEATURE_KEYS)}"
assert len(set(ALL_FEATURE_KEYS)) == 33, "Duplicate feature names detected!"

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Python        : {sys.version.split()[0]}")
print(f"✓ NumPy         : {np.__version__}")
print(f"✓ Feature groups: 4 modules → {len(ALL_FEATURE_KEYS)} features")
print(f"✓ CPU cores     : {os.cpu_count()}")

✓ Project root : c:\Users\USER\OneDrive\Máy tính\ai_detector_img
✓ Python        : 3.12.11
✓ NumPy         : 1.26.4
✓ Feature groups: 4 modules → 33 features
✓ CPU cores     : 12


## Cell 2 — System Configuration, Paths & Split Assignment

Reads the preprocessing manifest (`manifest.csv`) to obtain the authoritative list of `.npy` files  
with pre-attached `generator` and `label` metadata — no path-parsing heuristics needed.

**Split-role assignment** (prevents OOD leakage):

| Generator | Role | `split_role` |
|-----------|------|-------------|
| ADM, Midjourney, SDv14, VQDM, Wukong | In-Domain (5 ID) | `train_core` / `val` / `id_test` / `calibration` |
| SDv15, GLIDE | Out-of-Distribution (2 OOD) | `ood_eval` |

Split ratios for ID generators: **76% train_core → 4% calibration → 10% val → 10% id_test**  
(stratified by generator × label, deterministic seed=42).

In [8]:
"""Cell 2 — System Configuration, Paths & Split Assignment."""

# ── Paths ─────────────────────────────────────────────────────────────
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
MANIFEST_CSV   = PROCESSED_ROOT / "manifest.csv"
OUTPUT_CSV     = PROJECT_ROOT / "features" / "features_dataset.csv"

# ── Multiprocessing config ────────────────────────────────────────────
# Reserve 2 cores for OS + notebook kernel to avoid system freeze.
MAX_WORKERS = max(1, (os.cpu_count() or 4) - 2)

# Chunk size for ProcessPoolExecutor.map — balance IPC overhead vs memory.
# Each .npy is ~192 KB (256×256×3×float64), so 64 files ≈ 12 MB per chunk.
CHUNK_SIZE = 64

# ── Load manifest ────────────────────────────────────────────────────
assert MANIFEST_CSV.exists(), f"Manifest not found: {MANIFEST_CSV}"
manifest = pd.read_csv(MANIFEST_CSV)
manifest = manifest[manifest["action"] == "processed"].reset_index(drop=True)

# ══════════════════════════════════════════════════════════════════════
#  SPLIT_ROLE ASSIGNMENT — prevent OOD leakage
# ══════════════════════════════════════════════════════════════════════
ID_GENERATORS  = {"ADM", "Midjourney", "SDv14", "VQDM", "Wukong"}
OOD_GENERATORS = {"SDv15", "GLIDE"}
SPLIT_SEED     = 42

# Verify all generators are accounted for
all_gen = set(manifest["generator"].unique())
assert all_gen == ID_GENERATORS | OOD_GENERATORS, \
    f"Unknown generators: {all_gen - ID_GENERATORS - OOD_GENERATORS}"

# Step 1: OOD generators → ood_eval
manifest["split_role"] = ""
manifest.loc[
    manifest["generator"].isin(OOD_GENERATORS), "split_role"
] = "ood_eval"

# Step 2: ID generators → stratified train_core / val / id_test / calibration
#   Split unit : image-level, stratified by (generator × label)
#   Ratios     : 10% id_test, 10% val, then 5% of remaining → calibration,
#                rest → train_core
#   Note       : spec requests class-level split (§1.3 of p1.txt), but
#                ImageNet class labels for ILSVRC2012_val_* nature images
#                are not available in this workspace. Image-level stratified
#                split is used as a safe fallback. Refine to class-level when
#                ImageNet label mapping is integrated.

rng = np.random.default_rng(SPLIT_SEED)
id_mask = manifest["generator"].isin(ID_GENERATORS)
groups  = manifest[id_mask].groupby(["generator", "label"])

train_core_all, val_all, test_all, cal_all = [], [], [], []

for (_gen, _lab), grp in groups:
    idx = grp.index.to_numpy().copy()
    rng.shuffle(idx)
    n = len(idx)

    n_test       = round(n * 0.10)
    n_val        = round(n * 0.10)
    n_train_pool = n - n_test - n_val
    n_cal        = round(n_train_pool * 0.05)

    ptr = 0
    test_all.extend(idx[ptr : ptr + n_test]);            ptr += n_test
    val_all.extend(idx[ptr : ptr + n_val]);              ptr += n_val
    cal_all.extend(idx[ptr : ptr + n_cal]);              ptr += n_cal
    train_core_all.extend(idx[ptr :])

manifest.loc[test_all,       "split_role"] = "id_test"
manifest.loc[val_all,        "split_role"] = "val"
manifest.loc[cal_all,        "split_role"] = "calibration"
manifest.loc[train_core_all, "split_role"] = "train_core"

assert (manifest["split_role"] == "").sum() == 0, "Unassigned rows detected!"

# ── Provenance metadata ──────────────────────────────────────────────
manifest["dataset_name"]       = "GenImage"
manifest["preprocess_version"] = "v1"
manifest["feature_version"]    = "v1"

# ── Leakage guard (hard assert) ──────────────────────────────────────
ood_in_train = manifest[
    manifest["split_role"].isin(["train_core", "val", "calibration"])
    & manifest["generator"].isin(OOD_GENERATORS)
]
assert len(ood_in_train) == 0, \
    f"LEAKAGE DETECTED: {len(ood_in_train)} OOD rows in train/val/calibration!"

# ── Build task list ──────────────────────────────────────────────────
npy_paths  = manifest["output_path"].tolist()
generators = manifest["generator"].tolist()
labels     = manifest["label"].tolist()

# Spot-check: verify first file exists
assert Path(npy_paths[0]).exists(), f"First .npy not found: {npy_paths[0]}"

# ── Summary ──────────────────────────────────────────────────────────
print(f"✓ Manifest       : {len(manifest):,} processed files")
print(f"✓ Generators     : {manifest['generator'].nunique()} — {sorted(manifest['generator'].unique())}")
print(f"✓ Labels         : {sorted(manifest['label'].unique())}")
print(f"\n  Split distribution (seed={SPLIT_SEED}):")
for role in ["train_core", "calibration", "val", "id_test", "ood_eval"]:
    n = (manifest["split_role"] == role).sum()
    pct = 100 * n / len(manifest)
    print(f"    {role:14s}: {n:>6,}  ({pct:5.1f}%)")

id_total = id_mask.sum()
ood_total = len(manifest) - id_total
print(f"\n  ID total : {id_total:,}")
print(f"  OOD total: {ood_total:,}")

print(f"\n✓ Workers        : {MAX_WORKERS} processes (of {os.cpu_count()} cores)")
print(f"✓ Chunk size     : {CHUNK_SIZE}")
print(f"✓ Output         : {OUTPUT_CSV}")
print(f"✓ Leakage check  : 0 OOD rows in train/val/calibration — CLEAN")

✓ Manifest       : 87,971 processed files
✓ Generators     : 7 — ['ADM', 'GLIDE', 'Midjourney', 'SDv14', 'SDv15', 'VQDM', 'Wukong']
✓ Labels         : ['ai', 'nature']

  Split distribution (seed=42):
    train_core    : 45,590  ( 51.8%)
    calibration   :  2,400  (  2.7%)
    val           :  6,000  (  6.8%)
    id_test       :  6,000  (  6.8%)
    ood_eval      : 27,981  ( 31.8%)

  ID total : 59,990
  OOD total: 27,981

✓ Workers        : 10 processes (of 12 cores)
✓ Chunk size     : 64
✓ Output         : c:\Users\USER\OneDrive\Máy tính\ai_detector_img\features\features_dataset.csv
✓ Leakage check  : 0 OOD rows in train/val/calibration — CLEAN


## Cell 3 — The Unified Extractor

Single-file worker function designed for `ProcessPoolExecutor`:

1. **One disk read** per file (`np.load` with `allow_pickle=False`)
2. Feed the same array to all 4 extractors → **zero redundant I/O**
3. Per-file error isolation: exceptions → `status="error"` + NaN features
4. Returns a flat `dict` — serializable across process boundaries

In [9]:
"""Cell 3 — The Unified Extractor (validation)."""

# extract_all_features is imported from src.feature_extraction.worker (Cell 1).
# It reads one .npy from disk → feeds all 4 extractors → returns flat dict.
# Defined in a .py module (not inline) because Windows ProcessPoolExecutor
# requires worker functions to be importable.

# ── Quick validation: single file ────────────────────────────────────
_test_task = (npy_paths[0], generators[0], labels[0])
_test_result = extract_all_features(_test_task)
_finite = sum(1 for k in ALL_FEATURE_KEYS if np.isfinite(_test_result.get(k, np.nan)))

print(f"✓ Single-file test: {Path(_test_task[0]).name}")
print(f"  Status: {_test_result['status']}")
print(f"  Finite features: {_finite}/33")
for k in ALL_FEATURE_KEYS:
    v = _test_result[k]
    print(f"  {k:32s} = {v:.6e}" if np.isfinite(v) else f"  {k:32s} = NaN")

✓ Single-file test: 0_adm_153.npy
  Status: ok
  Finite features: 33/33
  frs_mid_variance                 = 1.030957e+00
  dct_mid_mean                     = 4.213208e+02
  dct_mid_variance                 = 9.033784e+05
  dct_mid_skewness                 = 6.085678e+00
  ps_alpha                         = 2.195683e+00
  ps_deviation_variance            = 7.398099e-03
  local_color_inconsistency        = 7.236375e-03
  pearson_y_cr                     = 6.157641e-02
  pearson_y_cb                     = -4.148620e-01
  pearson_cr_cb                    = 4.344206e-01
  energy_ratio_chroma              = 1.306637e-01
  glcm_contrast_cr                 = 3.231314e-02
  glcm_correlation_cr              = 8.565186e-01
  glcm_energy_cr                   = 7.436809e-01
  glcm_homogeneity_cr              = 9.838434e-01
  srm_square3_mar_cr               = 1.053863e+00
  srm_square3_energy_cr            = 2.369195e+00
  srm_edge3_mar_cr                 = 1.190294e+00
  srm_edge3_energy_cr      

## Cell 4 — Parallel Execution Engine

**Why `ProcessPoolExecutor`?**  
NumPy/SciPy operations (FFT, `convolve2d`, matrix algebra) release the GIL,  
but Python-level loops in feature extractors (DCT block iteration, GLCM computation, LBP scan) do **not**.  
`ThreadPoolExecutor` would serialize these → `ProcessPoolExecutor` gives true parallelism.

**IPC strategy:** Each worker returns a flat `dict` of Python floats (~1 KB per file) —  
minimal serialization overhead across process boundaries.

In [10]:
"""Cell 4 — Parallel Execution Engine."""


def run_parallel_extraction(
    tasks: list[tuple[str, str, str]],
    max_workers: int = MAX_WORKERS,
    chunk_size: int = CHUNK_SIZE,
) -> pd.DataFrame:
    """Extract features from all files using multiprocessing.

    Parameters
    ----------
    tasks : list of (npy_path, generator, label)
    max_workers : int
        Number of worker processes.
    chunk_size : int
        Batch size for ProcessPoolExecutor.map().

    Returns
    -------
    pd.DataFrame
        Columns: file_path, generator, label, 33 features, status, error.
    """
    n_total = len(tasks)
    records: list[dict] = []

    t_start = time.perf_counter()
    logger.info("Starting extraction: %d files × %d workers", n_total, max_workers)

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks — executor.map preserves order and uses chunksize
        # for efficient IPC batching
        results = executor.map(extract_all_features, tasks, chunksize=chunk_size)

        # Wrap with tqdm for progress tracking
        for record in tqdm(results, total=n_total, desc="Extracting features",
                           unit="img", smoothing=0.05):
            records.append(record)

    elapsed = time.perf_counter() - t_start
    rate = n_total / elapsed if elapsed > 0 else 0

    # ── Build DataFrame with deterministic column order ──────────────
    columns = ["file_path", "generator", "label"] + ALL_FEATURE_KEYS + ["status", "error"]
    df = pd.DataFrame(records, columns=columns)

    # ── Summary ──────────────────────────────────────────────────────
    n_ok = (df["status"] == "ok").sum()
    n_err = (df["status"] == "error").sum()

    print(f"\n{'='*60}")
    print(f"  Extraction complete")
    print(f"  Total files : {n_total:,}")
    print(f"  Success     : {n_ok:,} ({100*n_ok/n_total:.1f}%)")
    print(f"  Errors      : {n_err:,} ({100*n_err/n_total:.1f}%)")
    print(f"  Wall time   : {elapsed:.1f}s")
    print(f"  Throughput  : {rate:.0f} img/s")
    print(f"  Workers     : {max_workers}")
    print(f"{'='*60}")

    if n_err > 0:
        print(f"\n⚠ Error samples (first 5):")
        display(df[df["status"] == "error"][["file_path", "error"]].head())

    return df


# Prepare task list from manifest
tasks = list(zip(npy_paths, generators, labels))
print(f"✓ Task list prepared: {len(tasks):,} files")

✓ Task list prepared: 87,971 files


## Cell 5 — Execute, Merge Metadata & Save

Run the full extraction pipeline, merge `split_role` + provenance from the manifest,  
and persist to `features/features_dataset.csv`.  

The CSV uses `utf-8-sig` encoding for Excel compatibility on Windows.  
New columns (inserted before features): `split_role`, `dataset_name`, `preprocess_version`, `feature_version`.

In [11]:
"""Cell 5 — Execute, Merge Metadata & Save."""

# ══════════════════════════════════════════════════════════════════════
#  MAIN EXECUTION — Run the parallel extraction engine
# ══════════════════════════════════════════════════════════════════════

df_features = run_parallel_extraction(tasks)

# ── Merge split_role & provenance from manifest ──────────────────────
META_COLS = ["output_path", "split_role", "dataset_name",
             "preprocess_version", "feature_version"]
meta_df = manifest[META_COLS].rename(columns={"output_path": "file_path"})
df_features = df_features.merge(meta_df, on="file_path", how="left")

# Hard-assert: every row must have a split_role after merge
unmatched = df_features["split_role"].isna().sum()
assert unmatched == 0, f"MERGE ERROR: {unmatched} rows have no split_role!"

# ── Reorder columns: identity → provenance → features → status ──────
IDENTITY_COLS   = ["file_path", "generator", "label"]
PROVENANCE_COLS = ["split_role", "dataset_name", "preprocess_version", "feature_version"]
STATUS_COLS     = ["status", "error"]
columns = IDENTITY_COLS + PROVENANCE_COLS + ALL_FEATURE_KEYS + STATUS_COLS
df_features = df_features[columns]

# ── Persist to CSV ───────────────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_features.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✓ Saved: {OUTPUT_CSV}")
print(f"  Shape : {df_features.shape}")
print(f"  Size  : {OUTPUT_CSV.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Cols  : {list(df_features.columns[:7])} + {len(ALL_FEATURE_KEYS)} features + {STATUS_COLS}")

# ── Preview ──────────────────────────────────────────────────────────
display(df_features.head(3))

Extracting features:   0%|          | 0/87971 [00:00<?, ?img/s]


  Extraction complete
  Total files : 87,971
  Success     : 87,971 (100.0%)
  Errors      : 0 (0.0%)
  Wall time   : 1866.1s
  Throughput  : 47 img/s
  Workers     : 10

✓ Saved: c:\Users\USER\OneDrive\Máy tính\ai_detector_img\features\features_dataset.csv
  Shape : (87971, 42)
  Size  : 64.5 MB
  Cols  : ['file_path', 'generator', 'label', 'split_role', 'dataset_name', 'preprocess_version', 'feature_version'] + 33 features + ['status', 'error']


,file_path,generator,label,split_role,dataset_name,preprocess_version,feature_version,frs_mid_variance,dct_mid_mean,dct_mid_variance,...,spatial_snr_ratio,cross_noise_ratio,skew_noise_y,kurt_noise_y,skew_noise_cr,kurt_noise_cr,skew_noise_cb,kurt_noise_cb,status,error
0,C:\Users\USER\OneDrive\Máy tính\ai_detector_im...,ADM,ai,train_core,GenImage,v1,v1,1.030957,421.320827,9.033784e+05,...,1.058399,39.730771,-2.734414,58.094001,0.269196,2.854712,-0.852279,5.075478,ok,
1,C:\Users\USER\OneDrive\Máy tính\ai_detector_im...,ADM,ai,train_core,GenImage,v1,v1,0.852702,99.561968,2.519466e+05,...,0.588302,4.940439,0.004608,5.773345,-0.045126,0.731102,0.152121,0.192928,ok,
2,C:\Users\USER\OneDrive\Máy tính\ai_detector_im...,ADM,ai,train_core,GenImage,v1,v1,1.354846,475.052445,2.035070e+06,...,0.637323,74.221423,-2.163484,20.028583,-0.483847,6.747219,0.733725,4.235977,ok,


## Cell 6 — Split-Aware Sanity Checks

**Critical gate before model training.** All statistics-dependent checks (constant columns,
feature ranges) are computed **only on `train_core`** to prevent OOD leakage into any
downstream decision (imputation fit, feature selection, outlier thresholds).

| # | Check | Scope | Severity |
|---|-------|-------|----------|
| 1 | Row count matches manifest | Full | FATAL |
| 2 | Column schema integrity | Full | FATAL |
| 3 | Split-role distribution + leakage guard | Full | FATAL |
| 4 | Zero error rate | Full | WARNING |
| 5 | No all-NaN rows (except errors) | Full | FATAL |
| 6 | Group 1–3 guaranteed finite | Full | FATAL |
| 7 | **Constant columns (train_core only)** | train_core | WARNING |
| 8 | Group 4 NaN budget (Dual-Imputation) | Full | INFO |
| 9 | **Feature ranges (train_core only)** | train_core | WARNING |
| 10 | Class balance per split × generator | Full | INFO |

In [12]:
"""Cell 6 — Split-Aware Sanity Checks."""

print("=" * 65)
print("  POST-EXTRACTION SANITY CHECKS (split-aware)")
print("=" * 65)

df = df_features.copy()
ok_mask = df["status"] == "ok"
df_ok = df[ok_mask]
n_total = len(df)
n_ok = ok_mask.sum()
n_err = n_total - n_ok
passed = 0
total_checks = 10
feat_cols = ALL_FEATURE_KEYS

# Scope subsets
train_core_mask = df["split_role"] == "train_core"
df_train = df[train_core_mask & ok_mask]

# ── Check 1: Row count ──────────────────────────────────────────────
check1 = len(df) == len(manifest)
status1 = "PASS ✓" if check1 else "FAIL ✗"
print(f"\n[1/10] Row count: {len(df):,} vs manifest {len(manifest):,} → {status1}")
assert check1, f"Row mismatch: {len(df)} ≠ {len(manifest)}"
passed += 1

# ── Check 2: Column schema ──────────────────────────────────────────
expected_cols = (
    ["file_path", "generator", "label",
     "split_role", "dataset_name", "preprocess_version", "feature_version"]
    + ALL_FEATURE_KEYS
    + ["status", "error"]
)
check2 = list(df.columns) == expected_cols
status2 = "PASS ✓" if check2 else "FAIL ✗"
print(f"[2/10] Column schema: {len(df.columns)} columns → {status2}")
if not check2:
    missing = set(expected_cols) - set(df.columns)
    extra = set(df.columns) - set(expected_cols)
    if missing: print(f"       Missing: {missing}")
    if extra:   print(f"       Extra: {extra}")
assert check2, "Column schema mismatch"
passed += 1

# ── Check 3: Split-role distribution + leakage guard ─────────────────
print(f"[3/10] Split-role distribution & leakage guard:")
split_counts = df["split_role"].value_counts()
for role in ["train_core", "calibration", "val", "id_test", "ood_eval"]:
    n = split_counts.get(role, 0)
    pct = 100 * n / n_total
    print(f"       {role:14s}: {n:>6,}  ({pct:5.1f}%)")

# Hard leakage guard: no OOD generators in train/val/calibration
ood_leak = df[
    df["split_role"].isin(["train_core", "val", "calibration"])
    & df["generator"].isin({"SDv15", "GLIDE"})
]
check3 = len(ood_leak) == 0
status3 = "PASS ✓ (0 OOD in train/val/cal)" if check3 else f"FAIL ✗ ({len(ood_leak)} OOD leaked!)"
print(f"       Leakage guard: {status3}")
assert check3, f"LEAKAGE: {len(ood_leak)} OOD rows contaminate train/val/calibration!"

# Verify no unassigned rows
unassigned = (df["split_role"] == "").sum() + df["split_role"].isna().sum()
assert unassigned == 0, f"{unassigned} rows have no split_role!"
passed += 1

# ── Check 4: Error rate ─────────────────────────────────────────────
err_pct = 100 * n_err / n_total if n_total > 0 else 0
check4 = n_err == 0
status4 = "PASS ✓" if check4 else f"WARN ⚠ ({n_err:,} errors, {err_pct:.2f}%)"
print(f"[4/10] Error rate: {status4}")
if n_err > 0:
    err_by_gen = df[~ok_mask].groupby("generator").size()
    print(f"       Errors by generator:\n{err_by_gen.to_string()}")
passed += 1

# ── Check 5: No all-NaN rows among OK ───────────────────────────────
all_nan_ok = df_ok[feat_cols].isna().all(axis=1).sum()
check5 = all_nan_ok == 0
status5 = "PASS ✓" if check5 else f"FAIL ✗ ({all_nan_ok} all-NaN rows with status=ok!)"
print(f"[5/10] All-NaN rows (status=ok): {status5}")
assert check5, f"{all_nan_ok} rows have status=ok but all features are NaN!"
passed += 1

# ── Check 6: Group 1–3 guaranteed finite (contract) ─────────────────
group123_keys = list(FREQ_KEYS) + list(COLOR_KEYS) + list(MICRO_KEYS)
inf_count_123 = df_ok[group123_keys].apply(lambda s: (~np.isfinite(s)).sum()).sum()
check6 = inf_count_123 == 0
status6 = "PASS ✓" if check6 else f"FAIL ✗ ({inf_count_123} non-finite in Group 1-3!)"
print(f"[6/10] Group 1–3 all finite (status=ok): {status6}")
if not check6:
    for k in group123_keys:
        bad = (~np.isfinite(df_ok[k])).sum()
        if bad > 0:
            print(f"       {k}: {bad} non-finite")
assert check6, "Group 1–3 module contract violated: non-finite values detected"
passed += 1

# ── Check 7: Constant columns — TRAIN_CORE ONLY ─────────────────────
variances = df_train[feat_cols].var()
const_cols = variances[variances < 1e-15].index.tolist()
check7 = len(const_cols) == 0
status7 = "PASS ✓" if check7 else f"WARN ⚠ ({len(const_cols)} constant columns)"
print(f"[7/10] Constant columns (train_core only): {status7}")
if const_cols:
    for c in const_cols:
        print(f"       {c}: var={variances[c]:.2e}, val={df_train[c].iloc[0]}")
passed += 1

# ── Check 8: Group 4 NaN budget ─────────────────────────────────────
group4_keys = list(SPATIAL_KEYS)
nan_budget = {}
for k in group4_keys:
    n_nan = df_ok[k].isna().sum()
    nan_budget[k] = n_nan
print(f"[8/10] Group 4 NaN budget (Dual-Imputation):")
for k, v in nan_budget.items():
    pct = 100 * v / n_ok if n_ok > 0 else 0
    flag = "" if v == 0 else f"  ← {pct:.1f}% NaN (needs imputation)"
    print(f"       {k:32s}: {v:>6,} NaN{flag}")
passed += 1

# ── Check 9: Feature range — TRAIN_CORE ONLY ────────────────────────
print(f"[9/10] Feature statistics (train_core only, n={len(df_train):,}):")
stats = df_train[feat_cols].describe().T[["mean", "std", "min", "max"]]
stats["nan_count"] = df_train[feat_cols].isna().sum()
extreme_thresh = 1e6
extreme = stats[stats["max"].abs() > extreme_thresh]
if len(extreme) > 0:
    print(f"       ⚠ Features with |max| > {extreme_thresh:.0e}:")
    display(extreme)
else:
    print(f"       All features within reasonable range (|max| ≤ {extreme_thresh:.0e})")
display(stats)
passed += 1

# ── Check 10: Class balance per split × generator ────────────────────
print(f"[10/10] Balance per split_role × generator:")
balance = (
    df_ok.groupby(["split_role", "generator", "label"])
    .size()
    .unstack(fill_value=0)
)
balance["total"] = balance.sum(axis=1)
if "ai" in balance.columns and "nature" in balance.columns:
    balance["ai_pct"] = (balance["ai"] / balance["total"] * 100).round(1)
display(balance)
passed += 1

# ── Final verdict ────────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  SANITY CHECK SUMMARY: {passed}/{total_checks} checks passed")
if passed == total_checks:
    print(f"  ✓ Dataset is CLEAN — ready for model training")
    print(f"  ✓ OOD generators (SDv15, GLIDE) fully isolated in 'ood_eval'")
    print(f"  ✓ Feature statistics computed on train_core only (no leakage)")
else:
    print(f"  ⚠ Review warnings above before proceeding")
print(f"  Output: {OUTPUT_CSV}")
print(f"  Shape:  {df.shape}")
print(f"{'='*65}")

  POST-EXTRACTION SANITY CHECKS (split-aware)

[1/10] Row count: 87,971 vs manifest 87,971 → PASS ✓
[2/10] Column schema: 42 columns → PASS ✓
[3/10] Split-role distribution & leakage guard:
       train_core    : 45,590  ( 51.8%)
       calibration   :  2,400  (  2.7%)
       val           :  6,000  (  6.8%)
       id_test       :  6,000  (  6.8%)
       ood_eval      : 27,981  ( 31.8%)
       Leakage guard: PASS ✓ (0 OOD in train/val/cal)
[4/10] Error rate: PASS ✓
[5/10] All-NaN rows (status=ok): PASS ✓
[6/10] Group 1–3 all finite (status=ok): PASS ✓
[7/10] Constant columns (train_core only): PASS ✓
[8/10] Group 4 NaN budget (Dual-Imputation):
       spatial_snr_ratio               :     13 NaN  ← 0.0% NaN (needs imputation)
       cross_noise_ratio               :    850 NaN  ← 1.0% NaN (needs imputation)
       skew_noise_y                    :    265 NaN  ← 0.3% NaN (needs imputation)
       kurt_noise_y                    :    265 NaN  ← 0.3% NaN (needs imputation)
       skew_noi

,mean,std,min,max,nan_count
dct_mid_variance,3.358406e+06,1.197301e+07,0.0,7.076289e+08,0


,mean,std,min,max,nan_count
frs_mid_variance,1.594968e+00,6.862107e-01,0.000000,1.407508e+01,0
dct_mid_mean,3.177352e+02,3.701144e+02,0.000000,8.640988e+03,0
dct_mid_variance,3.358406e+06,1.197301e+07,0.000000,7.076289e+08,0
dct_mid_skewness,1.756569e+01,9.795127e+00,0.000000,1.547246e+02,0
ps_alpha,2.549420e+00,6.865991e-01,-3.222071,6.993486e+00,0
ps_deviation_variance,3.406721e-02,6.412022e-02,0.000000,2.633666e+00,0
local_color_inconsistency,2.835413e-02,4.057422e-02,0.000000,7.422883e-01,0
pearson_y_cr,-3.548057e-04,4.010715e-01,-0.998068,9.940290e-01,0
pearson_y_cb,-1.384612e-01,4.273672e-01,-0.999398,9.839046e-01,0
pearson_cr_cb,-6.087805e-01,4.169986e-01,-0.999276,9.987935e-01,0


[10/10] Balance per split_role × generator:


label                     ai  nature  total  ai_pct
split_role  generator                              
calibration ADM          240     240    480    50.0
            Midjourney   240     240    480    50.0
            SDv14        240     240    480    50.0
            VQDM         240     240    480    50.0
            Wukong       240     240    480    50.0
id_test     ADM          600     600   1200    50.0
            Midjourney   600     600   1200    50.0
            SDv14        600     600   1200    50.0
            VQDM         600     600   1200    50.0
            Wukong       600     600   1200    50.0
ood_eval    GLIDE       6000    5995  11995    50.0
            SDv15       7992    7994  15986    50.0
train_core  ADM         4560    4557   9117    50.0
            Midjourney  4560    4560   9120    50.0
            SDv14       4560    4556   9116    50.0
            VQDM        4560    4559   9119    50.0
            Wukong      4560    4558   9118    50.0
val         ADM          600     600   1200    50.0
            Midjourney   600     600   1200    50.0
            SDv14        600     600   1200    50.0
            VQDM         600     600   1200    50.0
            Wukong       600     600   1200    50.0


  SANITY CHECK SUMMARY: 10/10 checks passed
  ✓ Dataset is CLEAN — ready for model training
  ✓ OOD generators (SDv15, GLIDE) fully isolated in 'ood_eval'
  ✓ Feature statistics computed on train_core only (no leakage)
  Output: c:\Users\USER\OneDrive\Máy tính\ai_detector_img\features\features_dataset.csv
  Shape:  (87971, 42)
